# SleepSense — 01: Data Exploration (EDA)

**Goal:** Understand all data modalities, validate target variable distribution, identify coverage gaps.

## Confirmed CSV Schemas (from data inspection)
| File | Columns | Notes |
|------|---------|-------|
| `activity_uXX.csv` | `timestamp`, ` activity inference` | Has header; space in col name; 0=stationary,1=walking,2=running,3=unknown |
| `phonelock_uXX.csv` | `start`, `end` | Session intervals (not events); each row = one phone unlock session |
| `audio_uXX.csv` | `timestamp`, ` audio inference` | Has header; space in col name; 0=silence,1=voice,2=noise,3=unknown |
| `running_app_uXX.csv` | Various including `RUNNING_TASKS_topActivity_mPackage` | Has header |

## EMA JSON Schema
Each EMA response is a **flat dict**: `{resp_time, rate, hour, social, location}`
- `rate`: numeric string `'0'`=Very bad, `'1'`=Fairly bad, `'2'`=Fairly good, `'3'`=Very good
- `hour`: sleep duration in hours

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone

sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 4)

RAW  = PROJECT_ROOT / 'data' / 'raw'
IMPL = PROJECT_ROOT / 'implementation'
STUDY_START = 1362096000   # 2013-03-01 UTC
STUDY_END   = 1372636800   # 2013-07-01 UTC

def ts_to_date(ts):
    return datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime('%Y-%m-%d')

def ts_to_hour(ts):
    dt = datetime.fromtimestamp(int(ts), tz=timezone.utc)
    return dt.hour + dt.minute / 60.0

def load_activity(user_id):
    """Load activity CSV — has header, col name has leading space."""
    df = pd.read_csv(RAW / 'sensing' / 'activity' / f'activity_{user_id}.csv',
                     low_memory=False)
    df.columns = ['timestamp', 'activity_inference']
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    df['activity_inference'] = pd.to_numeric(df['activity_inference'], errors='coerce')
    df = df.dropna()
    df = df[(df['timestamp'] >= STUDY_START) & (df['timestamp'] <= STUDY_END)]
    return df

def load_phonelock(user_id):
    """Load phonelock sessions: start/end timestamps per unlock session."""
    df = pd.read_csv(RAW / 'sensing' / 'phonelock' / f'phonelock_{user_id}.csv',
                     low_memory=False)
    df.columns = ['start', 'end']
    df['start'] = pd.to_numeric(df['start'], errors='coerce')
    df['end'] = pd.to_numeric(df['end'], errors='coerce')
    df = df.dropna()
    df = df[(df['start'] >= STUDY_START) & (df['start'] <= STUDY_END)]
    df['duration_min'] = (df['end'] - df['start']) / 60
    df['start_hour'] = df['start'].apply(ts_to_hour)
    df['date'] = df['start'].apply(ts_to_date)
    return df

def load_audio(user_id):
    """Load audio CSV — has header, col name has leading space."""
    df = pd.read_csv(RAW / 'sensing' / 'audio' / f'audio_{user_id}.csv',
                     low_memory=False)
    df.columns = ['timestamp', 'audio_inference']
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    df['audio_inference'] = pd.to_numeric(df['audio_inference'], errors='coerce')
    df = df.dropna()
    df = df[(df['timestamp'] >= STUDY_START) & (df['timestamp'] <= STUDY_END)]
    return df

# All users
ACTIVITY_DIR = RAW / 'sensing' / 'activity'
ALL_USERS = sorted([f.stem.replace('activity_', '') for f in ACTIVITY_DIR.glob('activity_u*.csv')])
print(f'Found {len(ALL_USERS)} users: {ALL_USERS[:5]} ... {ALL_USERS[-3:]}')

## 2. Target Variable — EMA Sleep Quality

In [ ]:
RATE_SCORE_MAP = {'0': 0, '1': 1, '2': 2, '3': 3}
RATE_LABEL_MAP = {'0': 'Very bad', '1': 'Fairly bad', '2': 'Fairly good', '3': 'Very good'}
SLEEP_DIR = RAW / 'EMA' / 'response' / 'Sleep'

records = []
for uid in ALL_USERS:
    p = SLEEP_DIR / f'Sleep_{uid}.json'
    if not p.exists(): continue
    for rec in json.load(open(p)):
        if 'rate' not in rec: continue
        rate_str = str(rec['rate'])
        ts = rec.get('resp_time')
        try: ts_val = int(ts)
        except: ts_val = None
        try: hours = float(rec.get('hour', None))
        except: hours = None
        records.append({
            'user_id': uid,
            'timestamp': ts_val,
            'date': ts_to_date(ts_val) if ts_val else None,
            'rate_score': RATE_SCORE_MAP.get(rate_str),
            'rate_label': RATE_LABEL_MAP.get(rate_str, 'Unknown'),
            'sleep_hours': hours,
            'social': rec.get('social'),
        })

sleep_df = pd.DataFrame(records)
print(f'Total EMA sleep records: {len(sleep_df)}')
print(f'Users with data: {sleep_df["user_id"].nunique()}')
print(f'Score range: {sleep_df["rate_score"].min()} – {sleep_df["rate_score"].max()}')
print(f'\nRecords per user (describe):')
print(sleep_df.groupby('user_id').size().describe().round(1))
sleep_df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Label distribution
order = ['Very good', 'Fairly good', 'Fairly bad', 'Very bad']
lc = sleep_df['rate_label'].value_counts()
vals = [lc.get(o, 0) for o in order]
bars = axes[0].bar(order, vals, color=['#2ecc71','#3498db','#f39c12','#e74c3c'])
axes[0].set_title('EMA Sleep Quality Distribution (Target)')
axes[0].set_xlabel('Label')
axes[0].tick_params(axis='x', rotation=15)
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+1, str(v), ha='center', fontweight='bold')

# Records per user
rpu = sleep_df.groupby('user_id').size().sort_values()
axes[1].barh(rpu.index, rpu.values, color='#3498db')
axes[1].axvline(rpu.median(), color='red', linestyle='--', label=f'Median={rpu.median():.0f}')
axes[1].set_title('EMA Sleep Records per User')
axes[1].set_xlabel('# Records')
axes[1].legend()

# Sleep hours
axes[2].hist(sleep_df['sleep_hours'].dropna(), bins=20, color='#9b59b6', edgecolor='white')
axes[2].axvline(sleep_df['sleep_hours'].median(), color='red', linestyle='--',
                label=f'Median={sleep_df["sleep_hours"].median():.1f}h')
axes[2].set_title('Reported Sleep Duration')
axes[2].set_xlabel('Hours')
axes[2].legend()

plt.tight_layout()
plt.savefig(IMPL / 'eda_sleep_distribution.png', dpi=130, bbox_inches='tight')
plt.show()
print('Class balance (%):')
print((sleep_df['rate_label'].value_counts(normalize=True)*100).round(1))

## 3. Physical Activity

In [ ]:
ACTIVITY_LABELS = {0: 'Stationary', 1: 'Walking', 2: 'Running', 3: 'Unknown'}
sample_user = 'u00'

act_df = load_activity(sample_user)
print('Sample user shape:', act_df.shape)
print('Value counts:', act_df['activity_inference'].value_counts().to_dict())

activity_counts = {k: 0 for k in ACTIVITY_LABELS}
for uid in ALL_USERS:
    p = RAW / 'sensing' / 'activity' / f'activity_{uid}.csv'
    if not p.exists(): continue
    df = load_activity(uid)
    for k in ACTIVITY_LABELS:
        activity_counts[k] += int((df['activity_inference'] == k).sum())

fig, ax = plt.subplots(figsize=(8, 4))
labels = [ACTIVITY_LABELS[k] for k in sorted(activity_counts)]
values = [activity_counts[k] for k in sorted(activity_counts)]
bars = ax.bar(labels, values, color=['#e74c3c','#3498db','#2ecc71','#95a5a6'])
ax.set_title(f'Activity Distribution — All {len(ALL_USERS)} Users')
ax.set_ylabel('Total Samples')
for b, v in zip(bars, values):
    ax.text(b.get_x()+b.get_width()/2, v+max(values)*0.01, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()
total = sum(values)
print(f'Stationary: {activity_counts[0]/total:.1%} | Walking: {activity_counts[1]/total:.1%} | Running: {activity_counts[2]/total:.1%}')

## 4. Phone Lock (Session Data)

**Note:** `phonelock_uXX.csv` stores unlock *sessions* as `(start, end)` pairs — each row is one contiguous phone-use session.

In [ ]:
lock_df = load_phonelock(sample_user)
print('Phonelock sessions shape:', lock_df.shape)
print('Columns:', lock_df.columns.tolist())
print('\nSession duration stats (minutes):')
print(lock_df['duration_min'].describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Session start hour distribution
axes[0].hist(lock_df['start_hour'], bins=24, range=(0,24), color='#3498db', edgecolor='white')
axes[0].axvline(22, color='red', linestyle='--', label='22:00 threshold')
axes[0].set_xlabel('Session Start Hour')
axes[0].set_ylabel('# Sessions')
axes[0].set_title(f'Phone Sessions by Start Hour ({sample_user})')
axes[0].legend()

# Session duration distribution (cap at 120 min for visibility)
short = lock_df[lock_df['duration_min'] <= 120]['duration_min']
axes[1].hist(short, bins=40, color='#9b59b6', edgecolor='white')
axes[1].set_xlabel('Duration (minutes)')
axes[1].set_ylabel('# Sessions')
axes[1].set_title(f'Session Duration Distribution (≤120 min, {sample_user})')

# Daily session count
daily_sessions = lock_df.groupby('date').size()
axes[2].plot(range(len(daily_sessions)), daily_sessions.values, color='#e67e22', marker='o', ms=3)
axes[2].axhline(daily_sessions.mean(), color='red', linestyle='--', label=f'Mean={daily_sessions.mean():.1f}')
axes[2].set_xlabel('Study Day')
axes[2].set_ylabel('# Sessions')
axes[2].set_title(f'Daily Phone Sessions ({sample_user})')
axes[2].legend()

plt.tight_layout()
plt.show()

late_night = lock_df[lock_df['start_hour'] >= 22]
print(f'Late-night sessions (after 22:00): {len(late_night)} / {len(lock_df)} = {len(late_night)/len(lock_df):.1%}')

## 5. App Usage Data

In [ ]:
APP_CATS = {
    'social':        ['com.facebook','com.instagram','com.twitter','com.snapchat','com.whatsapp'],
    'entertainment': ['com.spotify','com.netflix','com.youtube','com.pandora'],
    'study':         ['edu.dartmouth','com.google.android.apps.docs'],
    'browser':       ['com.android.chrome','com.google.android.googlequicksearchbox'],
    'productivity':  ['com.google.android.gm','com.google.android.calendar'],
}
def cat_app(pkg):
    pkg = str(pkg)
    for c, ps in APP_CATS.items():
        if any(pkg.startswith(p) for p in ps): return c
    if pkg.startswith('com.android') or 'launcher' in pkg: return 'system'
    return 'other'

app_df = pd.read_csv(RAW / 'app_usage' / f'running_app_{sample_user}.csv', low_memory=False)
print('Columns:', app_df.columns.tolist()[:6], '...')
pkg_col = 'RUNNING_TASKS_topActivity_mPackage'
app_df['category'] = app_df[pkg_col].apply(cat_app)
app_df['timestamp'] = pd.to_numeric(app_df['timestamp'], errors='coerce')
app_df = app_df[(app_df['timestamp'] >= STUDY_START) & (app_df['timestamp'] <= STUDY_END)]
app_df['hour'] = app_df['timestamp'].apply(ts_to_hour)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
cc = app_df['category'].value_counts()
axes[0].bar(cc.index, cc.values, color=sns.color_palette('muted', len(cc)))
axes[0].set_title(f'App Category Distribution ({sample_user})')
axes[0].tick_params(axis='x', rotation=30)

hourly = app_df.assign(hb=app_df['hour'].astype(int)).groupby(['hb','category']).size().unstack(fill_value=0)
plot_cats = [c for c in ['social','entertainment','study','browser','productivity'] if c in hourly.columns]
if plot_cats:
    hourly[plot_cats].plot(ax=axes[1], colormap='tab10')
axes[1].axvline(22, color='red', linestyle='--', alpha=0.6, label='22:00')
axes[1].set_title('App Usage by Hour')
axes[1].set_xlabel('Hour')
axes[1].legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()
print('\nTop 10 packages:')
print(app_df[pkg_col].value_counts().head(10))

## 6. Audio Sensing

In [ ]:
AUDIO_LABELS = {0: 'Silence', 1: 'Voice', 2: 'Noise', 3: 'Unknown'}
aud = load_audio(sample_user)
print('Audio shape:', aud.shape)
print('Value counts:', aud['audio_inference'].value_counts().to_dict())

aud['hour'] = aud['timestamp'].apply(ts_to_hour)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

counts = aud['audio_inference'].value_counts()
labs = [AUDIO_LABELS.get(int(k), str(k)) for k in counts.index]
axes[0].bar(labs, counts.values, color=['#95a5a6','#2ecc71','#e67e22','#bdc3c7'][:len(labs)])
axes[0].set_title(f'Audio Inference ({sample_user})')
axes[0].set_ylabel('Count')

voice_h = aud[aud['audio_inference']==1].groupby(aud['hour'].astype(int)).size()
total_h = aud.groupby(aud['hour'].astype(int)).size()
vr = (voice_h / total_h).fillna(0)
axes[1].plot(vr.index, vr.values, color='#3498db', marker='o', ms=4)
axes[1].axvline(19, color='red', linestyle='--', alpha=0.6, label='19:00 evening')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Voice Fraction')
axes[1].set_title('Voice Detection Rate by Hour')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. PSQI Survey Baseline

In [ ]:
psqi_df = pd.read_csv(RAW / 'survey' / 'psqi.csv')
print('PSQI shape:', psqi_df.shape)
print('Columns:', psqi_df.columns.tolist())
print('\nTypes:', psqi_df['type'].unique() if 'type' in psqi_df.columns else 'no type col')
print('\nSample rows:')
print(psqi_df.head(10))

## 8. EMA Stress

In [ ]:
STRESS_DIR = RAW / 'EMA' / 'response' / 'Stress'

# Inspect schema
with open(STRESS_DIR / 'Stress_u00.json') as f:
    s0 = json.load(f)
all_keys = set()
[all_keys.update(r.keys()) for r in s0]
print('Stress JSON keys:', all_keys)
sample_with_data = [r for r in s0 if len(r) > 2]
print('Sample full records:', sample_with_data[:2])

# Build stress dataframe
STRESS_LABEL = {'1':'Feeling great','2':'Feeling good','3':'A little stressed',
                '4':'Definitely stressed','5':'Stressed out'}

srecs = []
for uid in ALL_USERS:
    p = STRESS_DIR / f'Stress_{uid}.json'
    if not p.exists(): continue
    for rec in json.load(open(p)):
        if 'level' not in rec: continue
        ts = rec.get('resp_time')
        lv = str(rec['level'])
        srecs.append({
            'user_id': uid,
            'timestamp': int(ts) if ts else None,
            'date': ts_to_date(int(ts)) if ts else None,
            'stress_level': int(lv) if lv.isdigit() else None,
            'stress_label': STRESS_LABEL.get(lv, f'Level {lv}')
        })

stress_df = pd.DataFrame(srecs)
print(f'\nStress records: {len(stress_df)} across {stress_df["user_id"].nunique() if len(stress_df) else 0} users')
if len(stress_df) > 0:
    print(stress_df['stress_level'].value_counts().sort_index())
    fig, ax = plt.subplots(figsize=(9, 4))
    vc = stress_df['stress_label'].value_counts()
    order_s = list(STRESS_LABEL.values())
    labs = [o for o in order_s if o in vc]
    ax.barh(labs, [vc[o] for o in labs],
            color=['#2ecc71','#3498db','#f39c12','#e67e22','#e74c3c'][:len(labs)])
    ax.set_title('EMA Stress Distribution')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No stress records found with level key — structure may differ')

## 9. Coverage & Alignment Check

In [ ]:
report = []
for uid in ALL_USERS:
    ema_dates = set(sleep_df[sleep_df['user_id'] == uid]['date'].dropna())
    act_path  = RAW / 'sensing' / 'activity' / f'activity_{uid}.csv'
    act_dates = set()
    if act_path.exists():
        df_a = load_activity(uid)
        act_dates = set(df_a['timestamp'].apply(ts_to_date))
    overlap = ema_dates & act_dates
    report.append({
        'user_id': uid,
        'ema_days': len(ema_dates),
        'activity_days': len(act_dates),
        'overlap_days': len(overlap),
        'pct': round(len(overlap)/len(ema_dates)*100, 1) if ema_dates else 0
    })

cov_df = pd.DataFrame(report)
print(cov_df.to_string(index=False))
print(f'\nTotal usable (user, day) pairs: {cov_df["overlap_days"].sum()}')
print(f'Users with < 10 labeled days: {(cov_df["ema_days"] < 10).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cov_s = cov_df.sort_values('overlap_days')
cols = ['#e74c3c' if v<10 else '#f39c12' if v<20 else '#2ecc71' for v in cov_s['overlap_days']]
axes[0].barh(cov_s['user_id'], cov_s['overlap_days'], color=cols)
axes[0].axvline(10, color='black', linestyle='--', alpha=0.5, label='10-day threshold')
axes[0].set_title('Usable (user, day) pairs')
axes[0].set_xlabel('# Days with EMA label + activity')
axes[0].legend()

axes[1].hist(cov_df['overlap_days'], bins=15, color='#3498db', edgecolor='white')
axes[1].axvline(cov_df['overlap_days'].median(), color='red', linestyle='--',
                label=f'Median={cov_df["overlap_days"].median():.0f}')
axes[1].set_title('Overlap Days Distribution')
axes[1].set_xlabel('# Usable Days')
axes[1].set_ylabel('# Users')
axes[1].legend()

plt.tight_layout()
plt.savefig(IMPL / 'eda_coverage.png', dpi=130, bbox_inches='tight')
plt.show()

## 10. Summary Report

In [ ]:
print('=' * 60)
print('SLEEPSENSE — EDA COMPLETE')
print('=' * 60)
print(f'Users: {len(ALL_USERS)}')
print(f'EMA sleep records: {len(sleep_df)} | Users with data: {sleep_df["user_id"].nunique()}')
print(f'Avg records/user: {len(sleep_df)/sleep_df["user_id"].nunique():.1f}')
print(f'Sleep score range: {sleep_df["rate_score"].min()}–{sleep_df["rate_score"].max()}')
print(f'Sleep hours: mean={sleep_df["sleep_hours"].mean():.1f}h, median={sleep_df["sleep_hours"].median():.1f}h')
print(f'\nTarget class balance:')
for lab, pct in (sleep_df['rate_label'].value_counts(normalize=True)*100).items():
    print(f'  {lab}: {pct:.1f}%')
print(f'\nTotal usable (user,day) pairs: {cov_df["overlap_days"].sum()}')
print(f'\nCONFIRMED SCHEMAS:')
print('  activity_uXX.csv  → timestamp, activity_inference [0-3]')
print('  phonelock_uXX.csv → start, end (session intervals in seconds)')
print('  audio_uXX.csv     → timestamp, audio_inference [0-3]')
print('  EMA Sleep JSON    → flat dict: resp_time, rate (str 0-3), hour, social')
print('  EMA Stress JSON   → flat dict: resp_time, level (str 1-5)')
print('=' * 60)